# Training on a public dataset: bev-decision-150K

[avbiswas/bev-decision-150K](https://huggingface.co/datasets/avbiswas/bev-decision-150K) holds
about 150,000 texts from 18 domains (spatial reasoning, sentiment, retail, support routing,
finance, ...), each with one or more typed questions and their correct answers. It is a much
harder test than the synthetic tickets in `quickstart.ipynb`: the questions differ from row to
row, and many need real reading comprehension.

This notebook trains Qwen3-0.6B with `tuning="head"` (frozen backbone, only the decision head
learns) and compares accuracy on held-out test rows **before** and **after** training, per
question type and per domain.

Running time scales with `N_TRAIN` and `N_TEST`: the defaults take about 10–15 minutes on a Mac.

In [ ]:
import json
import time
from collections import defaultdict

import pandas as pd
from datasets import load_dataset

from sysone import DecoderModel

N_TRAIN = 3000   # training rows (about 2 questions each); the full train split has 125,614
N_TEST = 1000    # held-out rows from the test split

## Load the data

Each row has a `state` (the text), a `questions_json` field with its questions, and a `domain`.
The correct answer is stored as a `label` inside each question, which sysone reads directly.
The domain is passed as the row's `category`, so validation is split evenly across domains.

In [ ]:
def load_rows(split, n):
    ds = load_dataset("avbiswas/bev-decision-150K", split=split).shuffle(seed=0).select(range(n))
    return [{"state": r["state"], "questions": json.loads(r["questions_json"]), "category": r["domain"]}
            for r in ds]


train = load_rows("train", N_TRAIN)
test = load_rows("test", N_TEST)

type_counts = pd.Series([q["type"] for r in test for q in r["questions"].values()]).value_counts()
print("%d training rows, %d test rows (%d test questions)" % (len(train), len(test), type_counts.sum()))
type_counts.rename("test questions")

In [ ]:
example = train[0]
print(example["category"], "|", example["state"][:400], "...")
next(iter(example["questions"].values()))

## Scoring

A choice is right if `choice` equals the label; a score is right if the expected level, rounded,
equals the label; a noul is right if `noul` (P(yes)) is on the label's side of 0.5.

Questions here have different numbers of options, so the baseline is **random guessing**: a
question with k options is guessed right 1/k of the time.

In [ ]:
TYPES = ("choice", "score", "noul")


def is_right(answer, label):
    if answer["type"] == "choice":
        return str(answer["choice"]) == str(label)
    if answer["type"] == "score":
        return round(answer["score"]) == int(label)
    return (answer["noul"] > 0.5) == bool(label)


def evaluate(model, rows):
    """Accuracy per question type, overall, and per domain."""
    preds = model.predict([{"state": r["state"], "questions": r["questions"]} for r in rows])
    by_type, by_domain = defaultdict(list), defaultdict(list)
    for row, pred in zip(rows, preds):
        for qid, answer in pred["answers"].items():
            hit = is_right(answer, row["questions"][qid]["label"])
            by_type[answer["type"]].append(hit)
            by_domain[row["category"]].append(hit)
    summary = {t: sum(by_type[t]) / len(by_type[t]) for t in TYPES}
    summary["overall"] = sum(sum(v) for v in by_type.values()) / sum(len(v) for v in by_type.values())
    return summary, {d: sum(v) / len(v) for d, v in by_domain.items()}


def random_guess(rows):
    by_type = defaultdict(list)
    for r in rows:
        for q in r["questions"].values():
            by_type[q["type"]].append(1 / (2 if q["type"] == "noul" else len(q["criteria"])))
    summary = {t: sum(by_type[t]) / len(by_type[t]) for t in TYPES}
    summary["overall"] = sum(sum(v) for v in by_type.values()) / sum(len(v) for v in by_type.values())
    return summary


results = {"random guess": random_guess(test)}
pd.DataFrame(results).T.round(3)

## Before training

sysone won't predict with an untrained head (it gives noise), so we fit one epoch with a learning
rate of 0 on a few rows. The head keeps its random initial weights.

In [ ]:
untrained = DecoderModel(tuning="head", epochs=1, head_lr=0.0, verbose=0).fit(train[:50])
results["before training"], _ = evaluate(untrained, test)
pd.DataFrame(results).T.round(3)

## Train

The backbone reads every training question once and its outputs are cached; each epoch then
trains only the head. 10% of rows are held back for validation and early stopping.

In [ ]:
start = time.time()
model = DecoderModel(tuning="head")
model.fit(train)
print("trained in %.1f minutes" % ((time.time() - start) / 60))
model.report_["validation"]

## After training

In [ ]:
results["after training"], after_by_domain = evaluate(model, test)
pd.DataFrame(results).T.round(3)

### Accuracy by domain

`n` is the number of test questions in that domain.

In [ ]:
counts = defaultdict(int)
for r in test:
    counts[r["category"]] += len(r["questions"])
by_domain = pd.DataFrame({"accuracy": after_by_domain, "n": counts}).sort_values("accuracy", ascending=False)
by_domain.round(3)

## What to expect

Training only the head on a frozen backbone is the cheapest setting and the least accurate. The
backbone's representations were never adapted to these questions, so expect clear gains over
random guessing but well short of what fine-tuning the backbone gives. Score questions are the
hardest: the loss treats levels as unrelated categories, so being one level off costs as much as
being three off.

Things to try:
- more training rows (`N_TRAIN`), which helps head-only training a lot;
- `num_layers` (default 0.7 of Qwen's 28 layers): middle layers sometimes carry more useful features;
- `tuning="lora"` / `"qlora"` once they are implemented.